# Cafe Sales — Data Cleaning Project



In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

RAW_PATH = "dirty_cafe_sales.csv"
CLEAN_PATH = "cafe_sales_cleaned.csv"

df_raw = pd.read_csv(RAW_PATH)
print(f"Shape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")
df_raw.head()


Shape: 10,000 rows x 8 columns


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


 Data Quality Report (on raw data)



In [2]:
print("dtypes (raw):")
print(df_raw.dtypes)
print()
print("Literal NaN count per column:")
print(df_raw.isna().sum())


dtypes (raw):
Transaction ID      object
Item                object
Quantity            object
Price Per Unit      object
Total Spent         object
Payment Method      object
Location            object
Transaction Date    object
dtype: object

Literal NaN count per column:
Transaction ID         0
Item                 333
Quantity             138
Price Per Unit       179
Total Spent          173
Payment Method      2579
Location            3265
Transaction Date     159
dtype: int64


In [3]:
# The raw nulls understate the real problem: 'ERROR' and 'UNKNOWN' are placeholder
# strings used throughout the dataset in place of missing values. Quantify them.
placeholder_values = ['ERROR', 'UNKNOWN']

placeholder_report = pd.DataFrame({
    'nan_count': df_raw.isna().sum(),
    'error_count': (df_raw == 'ERROR').sum(),
    'unknown_count': (df_raw == 'UNKNOWN').sum(),
})
placeholder_report['total_missing_equivalent'] = (
    placeholder_report['nan_count'] + placeholder_report['error_count'] + placeholder_report['unknown_count']
)
placeholder_report['pct_missing_equivalent'] = (
    placeholder_report['total_missing_equivalent'] / len(df_raw) * 100
).round(2)
placeholder_report


,nan_count,error_count,unknown_count,total_missing_equivalent,pct_missing_equivalent
Transaction ID,0,0,0,0,0.00
Item,333,292,344,969,9.69
Quantity,138,170,171,479,4.79
Price Per Unit,179,190,164,533,5.33
Total Spent,173,164,165,502,5.02
Payment Method,2579,306,293,3178,31.78
Location,3265,358,338,3961,39.61
Transaction Date,159,142,159,460,4.60


In [4]:
# Duplicate rows (exact full-row duplicates, including Transaction ID)
full_dupes = df_raw.duplicated().sum()

# Transaction ID should be a unique key -- check for duplicate IDs with differing data
id_dupes = df_raw.duplicated(subset=['Transaction ID']).sum()

print(f"Fully duplicated rows: {full_dupes}")
print(f"Duplicated Transaction IDs: {id_dupes}")


Fully duplicated rows: 0
Duplicated Transaction IDs: 0


In [5]:
# Value-range anomalies -- coerce numeric-looking columns to numeric (placeholders -> NaN)
# just for inspection purposes here; the real, permanent conversion happens in Section 3.
_qty = pd.to_numeric(df_raw['Quantity'], errors='coerce')
_price = pd.to_numeric(df_raw['Price Per Unit'], errors='coerce')
_total = pd.to_numeric(df_raw['Total Spent'], errors='coerce')
_dates = pd.to_datetime(df_raw['Transaction Date'], errors='coerce')

print("Quantity range:", _qty.min(), "-", _qty.max(), "| non-positive values:", (_qty <= 0).sum())
print("Price Per Unit range:", _price.min(), "-", _price.max(), "| non-positive values:", (_price <= 0).sum())
print("Total Spent range:", _total.min(), "-", _total.max(), "| non-positive values:", (_total <= 0).sum())
print("Transaction Date range:", _dates.min(), "-", _dates.max())
print("Unique Item values:", sorted(df_raw['Item'].dropna().unique()))
print("Unique Payment Method values:", sorted(df_raw['Payment Method'].dropna().unique()))
print("Unique Location values:", sorted(df_raw['Location'].dropna().unique()))


Quantity range: 1.0 - 5.0 | non-positive values: 0
Price Per Unit range: 1.0 - 5.0 | non-positive values: 0
Total Spent range: 1.0 - 25.0 | non-positive values: 0
Transaction Date range: 2023-01-01 00:00:00 - 2023-12-31 00:00:00
Unique Item values: ['Cake', 'Coffee', 'Cookie', 'ERROR', 'Juice', 'Salad', 'Sandwich', 'Smoothie', 'Tea', 'UNKNOWN']
Unique Payment Method values: ['Cash', 'Credit Card', 'Digital Wallet', 'ERROR', 'UNKNOWN']
Unique Location values: ['ERROR', 'In-store', 'Takeaway', 'UNKNOWN']


 Standardisation




In [6]:
df = df_raw.copy()

# Step 1: unify placeholders -> NaN across the whole frame
df = df.replace(['ERROR', 'UNKNOWN', 'error', 'unknown', ''], np.nan)

# Step 2: strip whitespace on all object columns
obj_cols = df.select_dtypes(include='object').columns
for c in obj_cols:
    df[c] = df[c].str.strip()

# Step 3: normalise categorical text casing defensively
for c in ['Item', 'Payment Method', 'Location']:
    df[c] = df[c].str.title()

print("Nulls after standardising placeholders:")
print(df.isna().sum())


Nulls after standardising placeholders:
Transaction ID         0
Item                 969
Quantity             479
Price Per Unit       533
Total Spent          502
Payment Method      3178
Location            3961
Transaction Date     460
dtype: int64


 Data Type Correction

In [7]:
df['Transaction ID'] = df['Transaction ID'].astype('string')

df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce').astype('Int64')
df['Price Per Unit'] = pd.to_numeric(df['Price Per Unit'], errors='coerce').astype('float64')
df['Total Spent'] = pd.to_numeric(df['Total Spent'], errors='coerce').astype('float64')
df['Transaction Date'] = pd.to_datetime(df['Transaction Date'], errors='coerce')

df['Item'] = df['Item'].astype('category')
df['Payment Method'] = df['Payment Method'].astype('category')
df['Location'] = df['Location'].astype('category')

df.dtypes


Transaction ID      string[python]
Item                      category
Quantity                     Int64
Price Per Unit             float64
Total Spent                float64
Payment Method            category
Location                  category
Transaction Date    datetime64[ns]
dtype: object

Missing Data Handling



In [8]:
# Confirm the price->item mapping and the qty*price=total identity on the cleaned data
price_item_map = (
    df.dropna(subset=['Item', 'Price Per Unit'])
      .groupby('Price Per Unit')['Item']
      .unique()
)
print("Price -> Item mapping:")
print(price_item_map)

item_price_map = (
    df.dropna(subset=['Item', 'Price Per Unit'])
      .groupby('Item')['Price Per Unit']
      .unique()
)
print()
print("Item -> Price mapping:")
print(item_price_map)


Price -> Item mapping:
Price Per Unit
1.0    ['Cookie']
Categories (8, object): ['Cake', 'C...
1.5    ['Tea']
Categories (8, object): ['Cake', 'Coff...
2.0    ['Coffee']
Categories (8, object): ['Cake', 'C...
3.0    ['Cake', 'Juice']
Categories (8, object): ['Ca...
4.0    ['Smoothie', 'Sandwich']
Categories (8, object...
5.0    ['Salad']
Categories (8, object): ['Cake', 'Co...
Name: Item, dtype: object

Item -> Price mapping:
Item
Cake        [3.0]
Coffee      [2.0]
Cookie      [1.0]
Juice       [3.0]
Salad       [5.0]
Sandwich    [4.0]
Smoothie    [4.0]
Tea         [1.5]
Name: Price Per Unit, dtype: object


C:\Users\karth\AppData\Local\Temp\ipykernel_23416\3230178461.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby('Item')['Price Per Unit']


In [9]:
mask_all3 = df[['Quantity', 'Price Per Unit', 'Total Spent']].notna().all(axis=1)
mismatches = (
    (df.loc[mask_all3, 'Quantity'] * df.loc[mask_all3, 'Price Per Unit'] -
     df.loc[mask_all3, 'Total Spent']).abs() > 0.01
).sum()
print(f"Rows with all of Quantity/Price/Total present: {mask_all3.sum()}")
print(f"Arithmetic mismatches among them: {mismatches}")


Rows with all of Quantity/Price/Total present: 8544
Arithmetic mismatches among them: 0


In [10]:
# --- Item: recover from a unique price, else mark Unknown Item ---
before_item_na = df['Item'].isna().sum()

price_to_single_item = {
    price: items[0] for price, items in price_item_map.items() if len(items) == 1
}

item_na_mask = df['Item'].isna()
recoverable_price_mask = item_na_mask & df['Price Per Unit'].isin(price_to_single_item.keys())
df.loc[recoverable_price_mask, 'Item'] = df.loc[recoverable_price_mask, 'Price Per Unit'].map(price_to_single_item)

df['Item'] = df['Item'].cat.add_categories(['Unknown Item'])
df['Item'] = df['Item'].fillna('Unknown Item')

print(f"Item: {before_item_na} missing -> recovered {recoverable_price_mask.sum()} from price, "
      f"{df['Item'].eq('Unknown Item').sum()} labelled 'Unknown Item'")


Item: 969 missing -> recovered 468 from price, 501 labelled 'Unknown Item'


In [10]:
# --- Quantity: recover as Total Spent / Price Per Unit ---
before_qty_na = df['Quantity'].isna().sum()

can_recover_qty = df['Quantity'].isna() & df['Total Spent'].notna() & df['Price Per Unit'].notna() & (df['Price Per Unit'] != 0)
implied_qty = (df.loc[can_recover_qty, 'Total Spent'] / df.loc[can_recover_qty, 'Price Per Unit'])
# only accept if it's (very close to) a whole number, matching how quantity actually behaves
whole_qty_mask = (implied_qty.round() - implied_qty).abs() < 0.01
recover_idx = implied_qty[whole_qty_mask].index
df.loc[recover_idx, 'Quantity'] = implied_qty.loc[recover_idx].round().astype('Int64')

print(f"Quantity: {before_qty_na} missing -> recovered {len(recover_idx)} algebraically, "
      f"{df['Quantity'].isna().sum()} still missing")


Quantity: 479 missing -> recovered 441 algebraically, 38 still missing


In [11]:
# --- Price Per Unit: recover as Total Spent / Quantity, else via Item -> Price map ---
before_price_na = df['Price Per Unit'].isna().sum()

can_recover_price = df['Price Per Unit'].isna() & df['Total Spent'].notna() & df['Quantity'].notna() & (df['Quantity'] != 0)
df.loc[can_recover_price, 'Price Per Unit'] = (
    df.loc[can_recover_price, 'Total Spent'] / df.loc[can_recover_price, 'Quantity']
)
recovered_from_total = can_recover_price.sum()

item_to_single_price = {item: prices[0] for item, prices in item_price_map.items() if len(prices) == 1}
still_na = df['Price Per Unit'].isna() & df['Item'].isin(item_to_single_price.keys())
df.loc[still_na, 'Price Per Unit'] = df.loc[still_na, 'Item'].map(item_to_single_price)
recovered_from_item = still_na.sum()

# last resort for any remainder: median price
remaining_na = df['Price Per Unit'].isna()
median_price = df['Price Per Unit'].median()
df.loc[remaining_na, 'Price Per Unit'] = median_price

print(f"Price Per Unit: {before_price_na} missing -> {recovered_from_total} from Total/Quantity, "
      f"{recovered_from_item} from Item lookup, {remaining_na.sum()} filled with median (${median_price:.2f})")


Price Per Unit: 533 missing -> 495 from Total/Quantity, 32 from Item lookup, 6 filled with median ($3.00)


In [12]:
# --- Total Spent: recompute from Quantity * Price Per Unit wherever possible ---
before_total_na = df['Total Spent'].isna().sum()

can_recompute_total = df['Total Spent'].isna() & df['Quantity'].notna() & df['Price Per Unit'].notna()
df.loc[can_recompute_total, 'Total Spent'] = (
    df.loc[can_recompute_total, 'Quantity'].astype('float64') * df.loc[can_recompute_total, 'Price Per Unit']
)

print(f"Total Spent: {before_total_na} missing -> recomputed {can_recompute_total.sum()}, "
      f"{df['Total Spent'].isna().sum()} still missing (will be dropped)")


Total Spent: 502 missing -> recomputed 482, 20 still missing (will be dropped)


In [13]:
# --- Payment Method / Location: explicit 'Missing' category, no imputation ---
for c in ['Payment Method', 'Location']:
    df[c] = df[c].cat.add_categories(['Missing'])
    df[c] = df[c].fillna('Missing')

print(df['Payment Method'].value_counts())
print()
print(df['Location'].value_counts())


Payment Method
Missing           3178
Digital Wallet    2291
Credit Card       2273
Cash              2258
Name: count, dtype: int64

Location
Missing     3961
Takeaway    3022
In-Store    3017
Name: count, dtype: int64


In [14]:
# --- Transaction Date: cannot be recovered -> row deletion as last resort ---
# --- Any remaining rows still missing Quantity or Total Spent are also dropped here ---
before_rows = len(df)

rows_before_date_drop = len(df)
df = df.dropna(subset=['Transaction Date'])
date_dropped = rows_before_date_drop - len(df)

rows_before_core_drop = len(df)
df = df.dropna(subset=['Quantity', 'Total Spent', 'Price Per Unit'])
core_dropped = rows_before_core_drop - len(df)

print(f"Rows dropped for missing Transaction Date: {date_dropped}")
print(f"Rows dropped for unrecoverable Quantity/Price/Total: {core_dropped}")
print(f"Rows remaining: {len(df)} (started cleaning with {before_rows})")
print()
print("Remaining nulls per column:")
print(df.isna().sum())


Rows dropped for missing Transaction Date: 460
Rows dropped for unrecoverable Quantity/Price/Total: 36
Rows remaining: 9504 (started cleaning with 10000)

Remaining nulls per column:
Transaction ID        0
Item                924
Quantity              0
Price Per Unit        0
Total Spent           0
Payment Method        0
Location              0
Transaction Date      0
dtype: int64


 Duplicate Removal



In [15]:
dupes_before = df.duplicated().sum()
id_dupes_before = df.duplicated(subset=['Transaction ID']).sum()
print(f"Fully duplicated rows: {dupes_before}")
print(f"Duplicated Transaction IDs: {id_dupes_before}")

rows_before_dedup = len(df)
df = df.drop_duplicates()
df = df.drop_duplicates(subset=['Transaction ID'], keep='first')
rows_removed_dupes = rows_before_dedup - len(df)

print(f"Duplicate rows removed: {rows_removed_dupes}")


Fully duplicated rows: 0
Duplicated Transaction IDs: 0
Duplicate rows removed: 0


Outlier Detection (IQR method)



In [16]:
def iqr_bounds(series):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

outlier_summary = {}
for col in ['Quantity', 'Price Per Unit', 'Total Spent']:
    s = df[col].astype('float64')
    low, high = iqr_bounds(s)
    n_outliers = ((s < low) | (s > high)).sum()
    outlier_summary[col] = {'lower_bound': low, 'upper_bound': high, 'n_outliers': n_outliers,
                             'min': s.min(), 'max': s.max()}

pd.DataFrame(outlier_summary).T


,lower_bound,upper_bound,n_outliers,min,max
Quantity,-1.0,7.0,0.0,1.0,5.0
Price Per Unit,-1.0,7.0,0.0,1.0,5.0
Total Spent,-8.0,24.0,259.0,1.0,25.0


In [17]:
# Verify the flagged 'outliers' are internally consistent, not corrupted
s = df['Total Spent']
low, high = iqr_bounds(s)
flagged = df[(s < low) | (s > high)]
consistent = ((flagged['Quantity'].astype('float64') * flagged['Price Per Unit'] - flagged['Total Spent']).abs() < 0.01).mean()
print(f"Share of flagged 'Total Spent' outliers that are arithmetically consistent: {consistent:.1%}")


Share of flagged 'Total Spent' outliers that are arithmetically consistent: 100.0%


## 7. Before vs. After Summary


In [18]:
def dtype_accuracy(frame, expected):
    correct = sum(1 for c, t in expected.items() if str(frame[c].dtype) == t)
    return f"{correct}/{len(expected)}"

expected_dtypes_before = {c: 'object' for c in df_raw.columns}
expected_dtypes_after = {
    'Transaction ID': 'string',
    'Item': 'category',
    'Quantity': 'Int64',
    'Price Per Unit': 'float64',
    'Total Spent': 'float64',
    'Payment Method': 'category',
    'Location': 'category',
    'Transaction Date': 'datetime64[ns]',
}

# raw null count should include placeholders to be a fair comparison
raw_missing_equiv = df_raw.replace(['ERROR', 'UNKNOWN'], np.nan).isna().sum().sum()

summary = pd.DataFrame({
    'Before': {
        'Row count': len(df_raw),
        'Null / missing-equivalent cells': raw_missing_equiv,
        'Fully duplicated rows': df_raw.duplicated().sum(),
        'Duplicate Transaction IDs': df_raw.duplicated(subset=['Transaction ID']).sum(),
        'Columns with correct dtype': dtype_accuracy(df_raw, expected_dtypes_before) + ' (all object)',
    },
    'After': {
        'Row count': len(df),
        'Null / missing-equivalent cells': df.isna().sum().sum(),
        'Fully duplicated rows': df.duplicated().sum(),
        'Duplicate Transaction IDs': df.duplicated(subset=['Transaction ID']).sum(),
        'Columns with correct dtype': dtype_accuracy(df, expected_dtypes_after) + ' / 8',
    }
})
summary


,Before,After
Row count,10000,9504
Null / missing-equivalent cells,10082,924
Fully duplicated rows,0,0
Duplicate Transaction IDs,0,0
Columns with correct dtype,8/8 (all object),8/8 / 8


In [19]:
print(f"Rows: {len(df_raw):,} -> {len(df):,}  ({len(df_raw) - len(df):,} removed, "
      f"{(len(df_raw)-len(df))/len(df_raw):.1%} of raw rows)")
print()
print("Final dtypes:")
print(df.dtypes)
print()
print("Final null counts (should be all zero):")
print(df.isna().sum())
df.head()


Rows: 10,000 -> 9,504  (496 removed, 5.0% of raw rows)

Final dtypes:
Transaction ID      string[python]
Item                      category
Quantity                     Int64
Price Per Unit             float64
Total Spent                float64
Payment Method            category
Location                  category
Transaction Date    datetime64[ns]
dtype: object

Final null counts (should be all zero):
Transaction ID        0
Item                924
Quantity              0
Price Per Unit        0
Total Spent           0
Payment Method        0
Location              0
Transaction Date      0
dtype: int64


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-Store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,4.0,Credit Card,In-Store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,Missing,Missing,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-Store,2023-06-11


Save Cleaned Dataset


In [20]:
df.to_csv(CLEAN_PATH, index=False)
print(f"Saved cleaned dataset to '{CLEAN_PATH}' -- {df.shape[0]:,} rows x {df.shape[1]} columns")


Saved cleaned dataset to 'cafe_sales_cleaned.csv' -- 9,504 rows x 8 columns


## Summary of Decisions

- Unified `"ERROR"` / `"UNKNOWN"` placeholder strings with true `NaN` across the board before
  any analysis, since treating them separately would have understated missingness and broken
  numeric/date conversion.
- Exploited two structural facts unique to this dataset — `Quantity x Price = Total Spent`
  and a (near) 1:1 `Item <-> Price` relationship — to recover `Item`, `Quantity`,
  `Price Per Unit`, and `Total Spent` algebraically wherever possible, instead of defaulting
  straight to mean/median/mode imputation. This preserves far more real transactions than a
  naive "impute everything" or "drop everything" approach.
- Left `Payment Method` and `Location` un-imputed (explicit `"Missing"` category) because
  nothing else in the row predicts them — fabricating a value would be a guess, not an
  inference.
- Dropped only the rows with an unrecoverable `Transaction Date` or unrecoverable
  Quantity/Price/Total combination, since date is required for time-series analysis and
  can't be inferred from anything else in the row.
- No true duplicate rows existed in this dataset (checked both before and after cleaning).
- Statistical outliers in `Total Spent` were investigated and found to be legitimate large
  orders (verified via the quantity x price identity), so they were retained rather than
  capped or removed.
